In [2]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [3]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-chat")

In [4]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [ ]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [ ]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [ ]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


## 1.2.使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [ ]:
# 打印结果
print(type(model))

# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [5]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

In [6]:
# 查看响应内容
print(response)

content='你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我是一个多模态模型，可以帮你解答各种问题、进行对话、处理文档等等。我的知识截止日期是2026年2月，拥有1M的超大上下文，可以一次性处理像《三体》三部曲那么大体量的内容。\n\n我支持：\n- 📝 文件上传（图像、PDF、Word、Excel、PPT等），可以读取其中的文字信息，也能识别图片中的内容\n- 🔗 阅读链接内容\n- 🌐 联网搜索（需要你在Web/App手动开启）\n- 🎤 语音输入（App端）\n\n而且我是**完全免费**的！你可以在官方应用商店下载App使用。\n\n有什么我可以帮你的吗？尽管问我吧！✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 6, 'total_tokens': 171, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'bc33bfd7-c9fb-490d-8fa0-85a2255cf7d9', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a099ce-e2fb-7ee1-adcc-016397f89f03-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'inp

In [7]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


我是迷人的反派角色，武藏！火箭队的一员，为了追寻强大的宝可梦而四处奔波。虽然每次都差一点点就成功了，但我们火箭队绝对不会放弃！喵喵和小次郎可是我的好搭档！


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [8]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [9]:
# 打印stream类型
print(type(stream))

<class 'generator'>


In [10]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个多模态模型，可以帮你回答问题、提供建议、进行创作等各种任务。我支持上传图像、PDF、Word、Excel等文件，可以从中提取文字信息进行处理，也能直接识别图片中的可见内容。

我的一些特点：
- **免费使用**，没有任何收费计划
- **上下文长度1M**，可以一次性处理超长文本（比如《三体》三部曲那么多内容）
- **支持联网搜索**（需要你手动开启）
- **支持语音输入**（App端）
- 知识截止日期到2026年2月

有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！🌟

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [11]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [12]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [13]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='0beba7a1-40d1-421e-ab68-89fa5a939387'), AIMessage(content='你好！我是 **DeepSeek**，由深度求索公司创造的AI助手。😊\n\n我是一个多模态模型，擅长回答各种问题、帮你分析信息、创作内容等等。我可以识别图片中的可见信息，也能读取你上传的文件（比如图片、PDF、Word、Excel等）中的文字内容来帮助你。\n\n我的一些特点：\n- **免费使用**，没有收费计划\n- **上下文长达1M**，可以一次性处理像《三体》三部曲那么大体量的内容\n- **支持联网搜索**（需要你手动开启）\n- **支持语音输入**（App端）\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，都可以问我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 6, 'total_tokens': 150, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'bee1d420-de8f-463b-bc8e-3e1759791ece', 'finish_reason': 'stop', 'lo

### 流式访问


In [14]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [15]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。😊

我可以帮你解答各种问题、进行对话、提供信息和建议。我支持上传文件（图片、PDF、Word、Excel 等）并从中读取文字信息，也能直接分析图片中的可见内容。我目前完全免费使用，上下文长度达到 1M，可以一次性处理像《三体》三部曲那样体量的书籍！

有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！